# Les députés représentent-ils bien leurs circonscriptions ?

*Introduction : ...*

# 1. Données

## 1.1 Sources

| Source | Contenu | Accès |
|---|---|---|
| INSEE, indicateurs par circonscription législative (2022) | 124 indicateurs socio-économiques | Fichier Excel téléchargé |
| INSEE, composition des circonscriptions | Correspondance communes - circonscriptions | Fichier Excel téléchargé |
| Députés actifs de l'Assemblée nationale | Âge, genre, groupe, scores parlementaires | API data.gouv.fr |

Les fichiers bruts sont « gelés » dans `data/data_brut` pour que le projet reste rejouable si les sites changent.

In [1]:
from extraction_donnees import charger_donnees
from jointures import joindre_deputes_insee
from descriptif import preparer_deputes

deputes, insee_indic, insee_compo, dico_insee = charger_donnees()
deputes = preparer_deputes(deputes)

print("Députés :", deputes.shape)
print("Indicateurs INSEE :", insee_indic.shape)
print("Composition communes-circonscriptions :", insee_compo.shape)

Députés : (574, 29)
Indicateurs INSEE : (567, 125)
Composition communes-circonscriptions : (35150, 8)


## 1.2 Nettoyage

Les données brutes n'étaient pas exploitables telles quelles :

- **Fichier INSEE des indicateurs** : les données sont dans le premier onglet, précédées de 7 lignes de métadonnées. La fonction `lire_excel_insee` prend donc l'onglet et la ligne d'en-tête en paramètres.
- **Clé de jointure `code_circo`** : l'INSEE utilise 5 caractères (`75018`), la base des députés les départements sur 3 chiffres et les circonscriptions sur 2. Nous avons harmonisé les deux au format INSEE.
- **Corse** : `2a001` côté députés, `2A001` côté INSEE, d'où un passage en majuscules.
- **Saint-Barthélemy** : l'INSEE la regroupe avec Saint-Martin (`97801`), le code `97701` a été rattaché à celle-ci.
- **Valeurs `s` et `nd`** de l'INSEE (secret statistique, non déterminé) : converties en valeurs manquantes.

In [2]:
df = joindre_deputes_insee(deputes, insee_indic)

print("Forme de la table jointe :", df.shape)
print(df["indicateur_present"].value_counts())
print(sorted(df.loc[~df["indicateur_present"], "code_circo"]))

Forme de la table jointe : (574, 154)
indicateur_present
True     563
False     11
Name: count, dtype: int64
['99001', '99002', '99003', '99004', '99005', '99006', '99007', '99008', '99009', '99010', '99011']


## 1.3 Jointure

La jointure est faite à gauche sur `code_circo` : sur 574 députés, **563 ont les indicateurs INSEE de leur circonscription**. Les 11 restants sont les circonscriptions des Français de l'étranger (`99001` à `99011`), pour lesquelles l'INSEE ne publie pas d'indicateurs. Nous les conservons avec des valeurs manquantes, car ils restent utiles pour les statistiques sur les députés eux-mêmes.